In [3]:
#tools creation
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun

In [4]:
#inbuilt tool
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki

WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\GENAI\\LANGCHAIN\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250))

In [5]:
wiki.name

'wikipedia'

In [6]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv

ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250))

In [7]:
arxiv.name

'arxiv'

In [8]:
#custom tools
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [10]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import openai
load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")
openai.api_key=os.getenv("OPENAI_API_KEY")

In [11]:
loader=WebBaseLoader(web_path=("https://docs.smith.langchain.com/")
                     )
docs=loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
documents=text_splitter.split_documents(docs)
vector_db=FAISS.from_documents(documents,OpenAIEmbeddings())
retriever=vector_db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002AF33479B40>, search_kwargs={})

In [12]:
from langchain.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-tool","serach any information regarding langsmith")
retriever_tool.name

'langsmith-tool'

In [13]:
tools=[wiki,arxiv,retriever_tool]

In [14]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\GENAI\\LANGCHAIN\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 Tool(name='langsmith-tool', description='serach any information regarding langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x000002AF334EEC20>, retriever=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002AF33479B40>, search_kw

In [15]:
import os
import openai
from dotenv import load_dotenv
load_dotenv()
## load the GROQ API Key
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [29]:
#from langchain_groq import ChatGroq
#llm=ChatGroq(groq_api_key=groq_api_key,model_name="Llama3-8b-8192")
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [18]:
#prompt
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")

In [19]:

from langchain.agents import AgentExecutor, create_openai_functions_agent

In [37]:
tools=[retriever_tool]

In [38]:
agent = create_openai_functions_agent(llm, tools, prompt)

In [39]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_function_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='H

In [40]:
agent_executor.invoke({"input": "what is LangChain?"})



> Entering new AgentExecutor chain...
LangChain is a powerful framework designed to simplify the development of applications that harness the capabilities of large language models (LLMs). It provides a comprehensive toolkit for building applications that can generate, analyze, and interpret natural language data. LangChain emphasizes several key areas:

1. **Model Interaction**: LangChain facilitates seamless interaction with language models, enabling developers to integrate these models into their applications easily.

2. **Data Augmentation**: It supports combining language models with other external data sources, such as APIs and databases, to enrich the application's functionality and output.

3. **Agent Systems**: LangChain supports the creation of agent systems where language models can be employed to perform decision-making tasks and act autonomously or semi-autonomously.

4. **Memory Management**: It offers capabilities to manage and store conversational history and other rel

{'input': 'what is LangChain?',
 'output': "LangChain is a powerful framework designed to simplify the development of applications that harness the capabilities of large language models (LLMs). It provides a comprehensive toolkit for building applications that can generate, analyze, and interpret natural language data. LangChain emphasizes several key areas:\n\n1. **Model Interaction**: LangChain facilitates seamless interaction with language models, enabling developers to integrate these models into their applications easily.\n\n2. **Data Augmentation**: It supports combining language models with other external data sources, such as APIs and databases, to enrich the application's functionality and output.\n\n3. **Agent Systems**: LangChain supports the creation of agent systems where language models can be employed to perform decision-making tasks and act autonomously or semi-autonomously.\n\n4. **Memory Management**: It offers capabilities to manage and store conversational history a